# 두 YOLO 모델 test 평가

두 모델은 이미 학습됐다. val 기준으로 YOLO11n (960)을 최종 모델로 미리 선택했다. test 결과는 최종 보고용이며 모델 선택이나 재학습에 사용하지 않는다.


In [1]:
!pip -q install ultralytics==8.4.208
import json, zipfile, yaml, torch, ultralytics
from pathlib import Path
from ultralytics import YOLO
from google.colab import drive
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요."
print("GPU:", torch.cuda.get_device_name(0), "Ultralytics:", ultralytics.__version__)
drive.mount("/content/drive")


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
GPU: Tesla T4 Ultralytics: 8.4.208
Mounted at /content/drive


## 데이터와 가중치 확인


In [2]:
DATA_ZIP = Path("/content/drive/MyDrive/construction-ppe-5class.zip")
DATA_PARENT = Path("/content/ppe_project_data")
DATA_ROOT = DATA_PARENT / "construction-ppe-5class"
assert DATA_ZIP.is_file(), f"데이터 ZIP 없음: {DATA_ZIP}"
if not (DATA_ROOT / "split_manifest.csv").is_file():
    DATA_PARENT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP) as archive:
        assert archive.testzip() is None, "ZIP이 손상되었습니다."
        archive.extractall(DATA_PARENT)
config = yaml.safe_load((DATA_ROOT / "data.yaml").read_text())
config["path"] = str(DATA_ROOT)
DATA_YAML = DATA_ROOT / "data_colab.yaml"
DATA_YAML.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))
assert len(list((DATA_ROOT / "images/test").glob("*.jpg"))) == 100
ROOT = Path("/content/drive/MyDrive/ppe-project")
WEIGHTS = {"yolo11n": ROOT / "runs/baseline_yolo11n_seed42/weights/best.pt",
           "yolo26n": ROOT / "runs/proposed_yolo26n_seed42/weights/best.pt",
           "yolo11n_960": ROOT / "runs/improved_yolo11n_img960_seed42/weights/best.pt"}
for name, path in WEIGHTS.items():
    assert path.is_file(), f"가중치 없음: {path}"
    print(name, path)


yolo11n /content/drive/MyDrive/ppe-project/runs/baseline_yolo11n_seed42/weights/best.pt
yolo26n /content/drive/MyDrive/ppe-project/runs/proposed_yolo26n_seed42/weights/best.pt
yolo11n_960 /content/drive/MyDrive/ppe-project/runs/improved_yolo11n_img960_seed42/weights/best.pt


## test 평가
세 모델에 동일한 test 100장을 적용한다. 입력 크기는 각 모델의 검증 설정을 따른다 (960 또는 640). 결과표와 혼동행렬은 Drive에 저장된다.


In [3]:
OUT = ROOT / "test_evaluation"
for name in WEIGHTS:
    if (OUT / name).exists():
        raise FileExistsError(f"기존 평가 결과가 있습니다: {OUT / name}")
OUT.mkdir(parents=True, exist_ok=True)
summary = {}
for name, weight in WEIGHTS.items():
    result = YOLO(str(weight)).val(data=str(DATA_YAML), split="test", imgsz=960 if name == "yolo11n_960" else 640, batch=8, device=0, plots=True, project=str(OUT), name=name, exist_ok=False)
    box = result.box
    summary[name] = {"precision": float(box.mp), "recall": float(box.mr), "mAP50": float(box.map50), "mAP50_95": float(box.map), "classes": {str(result.names[int(c)]): {"precision": float(box.p[i]), "recall": float(box.r[i]), "AP50": float(box.ap50[i]), "AP50_95": float(box.ap[i])} for i, c in enumerate(box.ap_class_index)}}
    (OUT / f"{name}_metrics.json").write_text(json.dumps(summary[name], ensure_ascii=False, indent=2))
(OUT / "comparison_metrics.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2))
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("저장 위치:", OUT)


Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,598,023 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3272.9±1082.7 MB/s, size: 470.9 KB)
val: Scanning /content/ppe_project_data/construction-ppe-5class/labels/test... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 1.1Kit/s 0.1s
val: New cache created: /content/ppe_project_data/construction-ppe-5class/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 2.4it/s 5.3s
                   all        100        698       0.93      0.756      0.842      0.622
                Person        100        235      0.946      0.892      0.943       0.78
                helmet         72        148       0.96      0.814      0.885      0.687
             no_helmet         49         85      0.869      0.553      0.674      0.387
               